In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments

/home/jayo/miniconda3/envs/llm_utils/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("burman-ai/The-Book-of-Mormon")

In [3]:
dataset

DatasetDict({
    train: Dataset({
        features: ['volume_id', 'book_id', 'chapter_id', 'verse_id', 'volume_title', 'book_title', 'volume_long_title', 'book_long_title', 'volume_subtitle', 'book_subtitle', 'volume_short_title', 'book_short_title', 'volume_lds_url', 'book_lds_url', 'chapter_number', 'verse_number', 'scripture_text', 'verse_title', 'verse_short_title'],
        num_rows: 5240
    })
})

In [4]:
# The dataset only comes with a train split, so let's add a test set
dataset = dataset["train"].train_test_split(test_size=0.1)

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['volume_id', 'book_id', 'chapter_id', 'verse_id', 'volume_title', 'book_title', 'volume_long_title', 'book_long_title', 'volume_subtitle', 'book_subtitle', 'volume_short_title', 'book_short_title', 'volume_lds_url', 'book_lds_url', 'chapter_number', 'verse_number', 'scripture_text', 'verse_title', 'verse_short_title'],
        num_rows: 4716
    })
    test: Dataset({
        features: ['volume_id', 'book_id', 'chapter_id', 'verse_id', 'volume_title', 'book_title', 'volume_long_title', 'book_long_title', 'volume_subtitle', 'book_subtitle', 'volume_short_title', 'book_short_title', 'volume_lds_url', 'book_lds_url', 'chapter_number', 'verse_number', 'scripture_text', 'verse_title', 'verse_short_title'],
        num_rows: 524
    })
})

In [6]:
dataset_train = dataset["train"]
dataset_test = dataset["test"]

In [7]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [8]:
def tokenize_function(examples):
    return tokenizer(examples["scripture_text"], truncation=True, padding="max_length")

In [9]:
dataset_train = dataset_train.map(tokenize_function, batched=True)
dataset_test = dataset_test.map(tokenize_function, batched=True)

Map: 100%|██████████| 524/524 [00:00<00:00, 2492.96 examples/s]


In [10]:
# The tokens are in 'input_ids' field. Also important is 'attention_mask'.
# Let's keep only those fields and discard the rest
dataset_train = dataset_train.remove_columns([col for col in dataset_train.column_names if col not in ["input_ids", "attention_mask"]])
dataset_test = dataset_test.remove_columns([col for col in dataset_test.column_names if col not in ["input_ids", "attention_mask"]])

In [11]:
# we add a labels field that is identical to input_ids for causal language modeling
# HF automatically shifts the labels over one position when computing the loss
dataset_train = dataset_train.map(
    lambda examples: {"labels": examples["input_ids"]},
    batched=True
)
dataset_test = dataset_test.map(
    lambda examples: {"labels": examples["input_ids"]},
    batched=True
)

Map: 100%|██████████| 524/524 [00:00<00:00, 1961.55 examples/s]


In [12]:
dataset_train

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4716
})

In [13]:
model = AutoModelForCausalLM.from_pretrained("gpt2")

In [14]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    fp16=True,
    logging_steps=100,
)

/home/jayo/miniconda3/envs/llm_utils/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/jayo/miniconda3/envs/llm_utils/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened

In [15]:
from transformers import TrainerCallback

class SampleGenerationCallback(TrainerCallback):
    def __init__(self, tokenizer, sample_prompt, max_new_tokens=50):
        self.tokenizer = tokenizer
        self.sample_prompt = sample_prompt
        self.max_new_tokens = max_new_tokens

    def generate_sample(self, model):
        model.eval()
        inputs = self.tokenizer(self.sample_prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=self.max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
        text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        print("\nSample output:\n", text)
        model.train()

    def on_train_begin(self, args, state, control, **kwargs):
        print(f"\n=== Starting training ===")
        self.generate_sample(kwargs["model"])

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None and "loss" in logs:
            print(f"\nStep {state.global_step} | Loss: {logs['loss']:.4f}")
            self.generate_sample(kwargs["model"])


In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_test,
    tokenizer=tokenizer,
    callbacks=[SampleGenerationCallback(tokenizer, sample_prompt="And it came to")]
)

/tmp/ipykernel_42384/3266332309.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [17]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.
wandb: Currently logged in as: jay-orten (jay-o) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



=== Starting training ===


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.



Sample output:
 And it came to pass that I was in the process of making a movie. When I was in college, I thought I was going to be a star, but I knew there wasn't a big budget for it. So I decided I'd give it a shot.


Step,Training Loss
100,1.158800
200,0.139700
300,0.146200
400,0.137800
500,0.144100
600,0.140300
700,0.139300
800,0.133100
900,0.132900
1000,0.127400



Step 100 | Loss: 1.1588

Sample output:
 And it came to pass that the Lord came down upon the people, saying, Go, and kill all the inhabitants of this land, and I will kill you.

Step 200 | Loss: 0.1397

Sample output:
 And it came to pass that the king of Babylon, having been a king of his own people, sent his sons to the king of Assyria. And he gave them the commandment that they should go into the land of Canaan and preach the gospel to the people.

Step 300 | Loss: 0.1462

Sample output:
 And it came to pass that I did make preparations for the coming of the Lord. And behold, when I had prepared my preparations, I did draw the Lord into my tent; and behold, the Lord stood upon the cross and bowed down upon the earth.

Step 400 | Loss: 0.1378

Sample output:
 And it came to pass that the Lord Jesus of Nazareth, the prophet of the law, and of the prophets of the law, and of the prophets of the law, and of the prophets of the law, and of the prophets of the law, and of the

Step 500 

TrainOutput(global_step=4716, training_loss=0.1535458838767779, metrics={'train_runtime': 1108.7799, 'train_samples_per_second': 4.253, 'train_steps_per_second': 4.253, 'total_flos': 2464506445824000.0, 'train_loss': 0.1535458838767779, 'epoch': 1.0})

In [18]:
trainer.push_to_hub("finetuned-gpt2-book-of-mormon")

Processing Files (2 / 2): 100%|██████████|  498MB /  498MB, 4.62MB/s  
New Data Upload: 100%|██████████|  498MB /  498MB, 4.62MB/s  


CommitInfo(commit_url='https://huggingface.co/royal42/results/commit/3433da108e881dbde610a80630e97c836c5c375f', commit_message='finetuned-gpt2-book-of-mormon', commit_description='', oid='3433da108e881dbde610a80630e97c836c5c375f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/royal42/results', endpoint='https://huggingface.co', repo_type='model', repo_id='royal42/results'), pr_revision=None, pr_num=None)